# [stand-in] CubeLang emitter — **gen 3** (v13e): the reverse-built free-text set, and the masking A/B

**What this is.** The v8e/v12e Unsloth LoRA SFT (LFM2.5-2.6B, r=32, 2 epochs, batch 32, lr 2e-4 cosine, MAX_SEQ
4096, `<think>` trained empty) on `emitter_sft_v13e.jsonl` = **gen 2 unchanged + the gen-3 records** that
`standin/data/build_gen3.py` built the reverse way (2026-09-13): a chain the store certifies first (the
encyclopedia's frame-read facts with their sentences, a seeded slice of the wiki world, two-hop chains off
those), a free-text wording of it second (`When was X born?`, `On what day, month, and year was X born?`,
`Where was X born?`, `Who is X married to?`, `When was the father of X born?`), and the plan the emitter must
learn is the **host's** -- the seed and the question's own relation words -- admitted only when the serving
gate itself certified it (coverage, typed answer class, resident VM, the VM's answer equal to the chain's
object). No LLM wrote any of it. A `plan` record (question alone → CotPlan) and a `chain` record (question +
Facts → the CotChain the walk built) per certified question, provenance on each.

**The A/B — one switch, `STANDIN_ARM`:**

- `masked` (default): the loss is taken on the **assistant response only** (Unsloth `train_on_responses_only`);
  the system prompt and the question are context, not targets. Nick's ask (2026-09-12): SFT with masking for gen 3.
- `full`: the v8e/v12e recipe as it was -- loss on the whole chat text.

Same data, same seed, same LoRA, same schedule; the adapter, merged model and GGUF land in an arm-named folder.
The read is not this notebook's exact-match smoke test; it is the gate on your machine: exp_r11 on the same 600
SimpleQA the earlier emitters ran (gen 2: 6 verified / 5 correct / 1 near / 0 wrong), and exp_r17 on the gen-3
**held** split (entities the training records never saw), scored verified / correct / **wrong** -- the last must
stay at zero, or the masking bought confidence without truth.

In [ ]:
# --- setup (run once per session) ---
import os, json, time, random, re
!pip -q install unsloth trl datasets
import sys
from google.colab import drive; drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cubbyllm/standin'
if not os.path.exists('/content/CubbyLLM/.git'):
    !rm -rf /content/CubbyLLM && git clone https://github.com/Grillcheese-AI/CubbyLLM.git /content/CubbyLLM 2>&1 | tail -2
else:
    !cd /content/CubbyLLM && git fetch origin 2>&1 | tail -1 && git reset -q --hard origin/master && git log -1 --format='repo at %h %s'
sys.path.insert(0, '/content/CubbyLLM/standin/data'); sys.path.insert(0, '/content/CubbyLLM')
try:
    from identity import EMITTER_SYSTEM
    print('identity: from the repo checkout')
except ImportError as e:
    assert os.path.exists(f'{DRIVE}/identity.py'), f'identity.py not in the repo checkout nor at {DRIVE} ({e})'
    sys.modules.pop('identity', None); sys.path.insert(0, DRIVE)
    from identity import EMITTER_SYSTEM
    print('identity: from Drive')
VERSION = os.environ.get('STANDIN_VERSION', 'v13e')    # gen 3 of the program emitter: gen 2 + the reverse-built free-text set
ARM = os.environ.get('STANDIN_ARM', 'masked')          # 'masked': loss on the assistant response only | 'full': the v8e/v12e recipe
assert ARM in ('masked', 'full'), ARM
DATA = f'{DRIVE}/emitter_sft_{VERSION}.jsonl'
MANIFEST = f'{DRIVE}/emitter_sft_{VERSION}.manifest.json'
OUT = f'{DRIVE}/emitter_lfm25_2p6b_{VERSION}'         # adapter + merged + GGUF land here
MODEL = os.environ.get('STANDIN_MODEL', 'LiquidAI/LFM2.5-2.6B')   # the v8e base; keep it, or the contrast is not attributable
MODEL_TAG = '' if MODEL == 'LiquidAI/LFM2.5-2.6B' else '_' + MODEL.split('/')[-1].lower().replace('.', 'p')
OUT = OUT + MODEL_TAG + '_' + ARM
print('arm:', ARM, '| out:', OUT)
EVAL_ONLY = os.environ.get('STANDIN_EVAL_ONLY') == '1'
if EVAL_ONLY: print('EVAL_ONLY: will load', f'{OUT}/merged', '(exists:', os.path.exists(f'{OUT}/merged'), ')')
MAX_SEQ = 4096   # the v8e setting (owner, 2026-09-03: 4096 trains better than 2048 on the A100-80G)
!nvidia-smi --query-gpu=name,memory.total --format=csv
for f in (DATA, MANIFEST):
    print(('ok      ' if os.path.exists(f) else 'MISSING ') + f)
m = json.load(open(MANIFEST))
assert m.get('version') == 'v13e' and 'gen3' in m, 'this is not a gen-3 manifest (build_gen3.py --merge writes emitter_sft_v13e.manifest.json)'
print('manifest', m['version'], ':', m['by_task'], '| records', m['n_records'], '| train rows after repeat', m['train_rows_after_repeat'])
print('by source:', m['by_source'])

In [ ]:
# --- data: chat-format the records; train/val from the builder's deterministic split ---
SYSTEM = EMITTER_SYSTEM
from collections import Counter
recs = [json.loads(l) for l in open(DATA, encoding='utf-8')]
# the v8e filter (VM-ok, and not gold-wrong) -- EXCEPT the r7 chains, which enter on the VM's word alone:
# gold is not available at serve time, and filtering by it would make gen 2's data cleaner than the loop
# can ever produce. How many are gold-wrong is printed from the manifest above.
R7 = 'cubbyllm/cot_harvest_r7'
recs = [r for r in recs if r.get('vm_ok') in (True, None) and (r.get('gold_match') is not False or r.get('source') == R7)]
# gen 3's records carry the gate's verdict (vm_ok True on the chain record, gold_match True: the VM's answer IS the chain's
# object); their 'held' split is entities no training record uses -- the val split here, and exp_r17's held-out set
train = [r for r in recs if r['split'] == 'train']; val = [r for r in recs if r['split'] in ('val', 'held')]
print('by source (train):', Counter(r.get('source') for r in train).most_common())
print('train', len(train), Counter(r['task'] for r in train)); print('val  ', len(val), Counter(r['task'] for r in val))
print('r7 chains in train:', sum(r.get('source') == R7 for r in train), '| plan records in train:', sum(r['task'] == 'plan' for r in train))

NO_THINK = '<think>\n</think>\n'   # LFM2.5 opens <think> on its own; train it to close immediately (v2, 2026-08-30)
def to_messages(r):
    return [{'role': 'system', 'content': r.get('system') or SYSTEM},
            {'role': 'user', 'content': r['prompt']},
            {'role': 'assistant', 'content': NO_THINK + r['program'].strip() + '\n'}]

from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained((f'{OUT}/merged' if EVAL_ONLY else MODEL), max_seq_length=MAX_SEQ, load_in_4bit=False, dtype=None)
print('model:', f'{OUT}/merged (trained, from Drive)' if EVAL_ONLY else MODEL)

def fmt(r):
    return {'text': tokenizer.apply_chat_template(to_messages(r), tokenize=False, add_generation_prompt=False)}
from datasets import Dataset
random.Random(0).shuffle(train)
ds_train = Dataset.from_list([fmt(r) for r in train for _ in range(int(r.get('repeat', 1)))])   # builder's `repeat`: chains x3, r7 x6
print('train rows after repeat weights:', len(ds_train))
lens = [len(tokenizer(x['text']).input_ids) for x in ds_train.select(range(min(500, len(ds_train))))]
print('token lengths (sample of 500): max', max(lens), 'p95', sorted(lens)[int(0.95*len(lens))], '-> MAX_SEQ', MAX_SEQ)
plan_example = next(r for r in train if r['task'] == 'plan')
print('\na plan record, formatted:\n', fmt(plan_example)['text'][:700])

In [ ]:
# --- LoRA + SFT: the v8e settings, unchanged ---
if EVAL_ONLY:
    print('EVAL_ONLY: skipping LoRA + training; the merged model from Drive is already loaded')
else:
    from trl import SFTTrainer, SFTConfig
    model = FastLanguageModel.get_peft_model(
        model, r=32, lora_alpha=32, lora_dropout=0.0, bias='none',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'out_proj', 'o_proj', 'in_proj', 'w1', 'w2', 'w3', 'gate_proj', 'up_proj', 'down_proj'],
        use_gradient_checkpointing='unsloth', random_state=0)
    cfg = SFTConfig(output_dir='/content/emitter_ckpt', per_device_train_batch_size=32, gradient_accumulation_steps=1,
                    num_train_epochs=2, learning_rate=2e-4, lr_scheduler_type='cosine', warmup_steps=20,
                    logging_steps=10, save_strategy='no', bf16=True, max_seq_length=MAX_SEQ, dataset_text_field='text',
                    packing=False, report_to='none', seed=0)
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds_train, args=cfg)
    if ARM == 'masked':
        # loss on the assistant response only: the chat template's user / assistant headers, read off a
        # rendered probe rather than assumed (LFM2.5 is ChatML-shaped: <|im_start|>user ... <|im_start|>assistant)
        from unsloth.chat_templates import train_on_responses_only
        probe = tokenizer.apply_chat_template([{'role': 'user', 'content': 'XQX'}, {'role': 'assistant', 'content': 'YQY'}], tokenize=False)
        i, j = probe.find('XQX'), probe.find('YQY'); k = probe.rfind('<|im_start|>', 0, i)
        INSTR = probe[k:i]; between = probe[i + 3:j]; RESP = between[between.rfind('<|im_start|>'):] if '<|im_start|>' in between else between
        assert INSTR and RESP and INSTR != RESP, (INSTR, RESP)
        print('masking: instruction_part', repr(INSTR), '| response_part', repr(RESP))
        trainer = train_on_responses_only(trainer, instruction_part=INSTR, response_part=RESP)
        ex = trainer.train_dataset[0]; n_lab = sum(1 for x in ex['labels'] if x != -100); n_tok = len(ex['labels'])
        assert 0 < n_lab < n_tok, (n_lab, n_tok)
        print(f'masking check on row 0: {n_lab}/{n_tok} tokens carry loss ({n_lab / n_tok:.0%}); the rest is context')
        print('  loss tokens decode to:', repr(tokenizer.decode([t for t in ex['labels'] if t != -100])[:200]))
    else:
        print('full: loss on the whole chat text, as v8e / v12e')
    t0 = time.time(); stats = trainer.train(); print(f'trained in {(time.time()-t0)/60:.1f} min; final loss', stats.training_loss)
    os.makedirs(OUT, exist_ok=True); model.save_pretrained(f'{OUT}/adapter'); tokenizer.save_pretrained(f'{OUT}/adapter')
    json.dump({'version': VERSION, 'arm': ARM, 'base': MODEL, 'gen2': m.get('gen2'), 'gen3': m.get('gen3'),
               'train_rows': len(ds_train), 'final_loss': stats.training_loss,
               'lora': {'r': 32, 'alpha': 32, 'targets': 'q,k,v,out,o,in_proj,w1,w2,w3,gate,up,down'},
               'sft': {'epochs': 2, 'batch': 32, 'lr': 2e-4, 'sched': 'cosine', 'warmup': 20, 'max_seq': MAX_SEQ, 'seed': 0}},
              open(f'{OUT}/train_card.json', 'w'), indent=1)
    print('adapter ->', f'{OUT}/adapter', '| train card ->', f'{OUT}/train_card.json')

In [ ]:
# --- export FIRST: merged fp16 + GGUF (q8_0 for the VM eval, q4_k_m for the 12 GB Vulkan box) ---
if EVAL_ONLY:
    print('EVAL_ONLY: skipping export; the merged model from Drive is already loaded')
else:
    model.save_pretrained_merged(f'{OUT}/merged', tokenizer, save_method='merged_16bit')
    model.save_pretrained_gguf(f'{OUT}/gguf', tokenizer, quantization_method=['q8_0', 'q4_k_m'])
    import glob as _glob
    GGUFS = sorted(_glob.glob(f'{OUT}/gguf*/*.gguf'))
    print('GGUF files:'); [print('  ', g, f'{os.path.getsize(g)/1e9:.2f} GB') for g in GGUFS]
    print(f'\nlocally, as standin/models/emitter_v13e_{ARM}.Q4_K_M.gguf, the gate:')
    print(f'  python validation/exp_r11_search_learn.py --wikidata --n 600 --seed 7 --lexicon --gguf standin/models/emitter_v13e_{ARM}.Q4_K_M.gguf --tag _gen3_{ARM}')
    print(f'  python validation/exp_r17_gen3_heldout.py --gguf standin/models/emitter_v13e_{ARM}.Q4_K_M.gguf --tag _{ARM}')

In [ ]:
# --- OPTIONAL format-level smoke eval on the val split (the verified read is local: eval_emitter_vm.py + exp_r3/exp_r7) ---
# `plan` is scored by exact match on the CotPlan text (deterministic given the question), and ALSO by
# "role chain matches" -- the relations and their order, ignoring the seed spelling -- which is the part the
# disposer reads. Set STANDIN_EVAL_N=0 to skip.
FastLanguageModel.for_inference(model)
import transformers; transformers.logging.set_verbosity_error()
def strip_think(s):
    return re.sub(r'^\s*(?:<think>)?.*?</think>\s*', '', s, count=1, flags=re.S) if '</think>' in s else s
def norm(s): return re.sub(r'\s+', ' ', re.sub(r'#.*', '', strip_think(s))).strip()
BINDS = re.compile(r'bind\s+frame\s*,\s*([A-Za-z_][A-Za-z0-9_]*)\s*,\s*"((?:[^"\\]|\\.)*)"\s*;')
def role_chain(prog):   # CotPlan: the HOPk strings in order; CotChain: the H-roles in order
    b = BINDS.findall(strip_think(prog))
    hops = sorted((int(k[3:]), v.lower()) for k, v in b if k.startswith('HOP') and k[3:].isdigit())
    if hops: return [v for _, v in hops]
    return [k for k, _ in b if re.match(r'^H\d+_', k)]
def emit(prompt, max_new=768, system=None):
    text = tokenizer.apply_chat_template([{'role': 'system', 'content': system or SYSTEM}, {'role': 'user', 'content': prompt}],
                                         tokenize=False, add_generation_prompt=True) + NO_THINK
    enc = tokenizer(text, return_tensors='pt', add_special_tokens=False).to('cuda')
    out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
N_PER_TASK = int(os.environ.get('STANDIN_EVAL_N', '8'))
rng = random.Random(1); by_task = {}
for r in val: by_task.setdefault(r['task'], []).append(r)
sample = [r for t, rs in sorted(by_task.items()) for r in rng.sample(rs, min(N_PER_TASK, len(rs)))]
CAP = {'chain': 500, 'plan': 300, 'kernel': 600, 'arithmetic': 500, 'role_binding': 400}
print('eval sample', len(sample), {t: min(N_PER_TASK, len(rs)) for t, rs in sorted(by_task.items())})
hits = Counter(); tot = Counter(); chain_ok = Counter(); outputs = []
t0 = time.time()
for i, r in enumerate(sample):
    gen = emit(r['prompt'], max_new=CAP.get(r['task'], 600)); ok = norm(gen) == norm(r['program'])
    tot[r['task']] += 1; hits[r['task']] += int(ok)
    rc = None
    if r['task'] in ('plan', 'chain'):
        rc = role_chain(gen) == role_chain(r['program']); chain_ok[r['task']] += int(rc)
    outputs.append({'id': r['id'], 'task': r['task'], 'subtype': r.get('subtype', ''), 'source': r.get('source'),
                    'prompt': r['prompt'], 'reference': r['program'], 'generated': gen, 'exact_match': ok,
                    'role_chain_match': rc, 'gold': r.get('gold')})
    if (i+1) % 25 == 0: print(f'  {i+1}/{len(sample)} ({time.time()-t0:.0f}s)')
print('[stand-in] val exact-match by task:', {t: f'{hits[t]}/{tot[t]}' for t in tot}, '| overall', round(sum(hits.values())/max(1, sum(tot.values())), 3))
print('[stand-in] role-chain match (relations + order, what the disposer reads):', {t: f'{chain_ok[t]}/{tot[t]}' for t in chain_ok})
json.dump({'model': MODEL, 'version': VERSION, 'n': len(sample), 'exact_match_by_task': {t: hits[t]/tot[t] for t in tot},
           'role_chain_by_task': {t: chain_ok[t]/tot[t] for t in chain_ok}, 'outputs': outputs,
           'manifest_output_sha256': m['output_sha256']}, open(f'{OUT}/val_generations.json', 'w'), indent=1)
print('generations ->', f'{OUT}/val_generations.json')


### How to read

- **This notebook proves nothing about the gate.** The smoke eval is a format read. The gate is on your machine:
  `python validation/exp_r11_search_learn.py --wikidata --n 600 --seed 7 --lexicon --gguf standin/models/emitter_v13e_<arm>.Q4_K_M.gguf --tag _gen3_<arm>`
  (the same 600 SimpleQA as gen 2's `_local` run: 6 verified, 5 correct, 1 near, 0 wrong, 0 API calls), and
  `python validation/exp_r17_gen3_heldout.py --gguf standin/models/emitter_v13e_<arm>.Q4_K_M.gguf --tag _<arm>`
  (the gen-3 **held** split: free-text wordings over entities no training record used).
- **What to compare, masked vs full, on the same questions:** plans the disposer accepts; VM-verified and
  correct; and the honest one, **wrong**, which must not rise. Then gen 2 → gen 3 on SimpleQA: more verified at
  0 wrong is the free-text hole closing; more verified with wrong rising is the loop teaching confident wrong
  plans, and the disposer -- not the emitter -- is what to look at next.
- **Everything here is `[stand-in]`.** It goes in `standin/README.md` and the research doc, never in
  `CUBBYLLM_HYPOTHESES.md` except as a pointer.